<a href="https://colab.research.google.com/github/agnishreddy/-Development-of-a-High-Affinity-Peptidomimetic-Inhibitor-Targeting-SARS-CoV-Main-Protease-M-pro-/blob/main/Development_of_a_High_Affinity_Peptidomimetic_Inhibitor_Targeting_SARS_CoV_Main_Protease_(Mpro).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Install Conda-colab to handle complex structural biology packages
!pip install -q condacolab
import condacolab
condacolab.install()

# Install AutoDock Vina, Meeko, and OpenBabel using conda
!conda install -c conda-forge vina meeko openbabel -y
# Install RDKit, chembl_webresource_client and other packages using pip into the active Python environment
import sys
!{sys.executable} -m pip install rdkit chembl_webresource_client lightgbm xgboost openpyxl

✨🍰✨ Everything looks OK!
Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / - \ | / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 56.3 MB/s eta 0:00:00


In [7]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

# Target search
target = new_client.target
target_query = target.search('coronavirus')
targets = pd.DataFrame.from_dict(target_query)

# Select SARS-CoV-2 3C-like proteinase (Target ID: CHEMBL4495581 or similar)
selected_target = 'CHEMBL3927' # Example single protein target

# Fetch bioactivity data
activity = new_client.activity
res = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")
df = pd.DataFrame.from_dict(res)

# Filter out rows missing SMILES string or IC50 values
df = df[df.canonical_smiles.notna() & df.standard_value.notna()]
df.to_csv('bioactivity_data_raw.csv', index=False)
print(f"Retrieved {len(df)} bioactivity compounds.")


Retrieved 262 bioactivity compounds.


In [12]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

def calculate_lipinski_safe(smiles_list):
    mw, logp, h_donors, h_acceptors, valid_indices = [], [], [], [], []

    for idx, s in enumerate(smiles_list):
        try:
            mol = Chem.MolFromSmiles(str(s))
            if mol is not None:
                mw.append(Descriptors.MolWt(mol))
                logp.append(Descriptors.MolLogP(mol))
                h_donors.append(Lipinski.NumHDonors(mol))
                h_acceptors.append(Lipinski.NumHAcceptors(mol))
                valid_indices.append(idx)
        except Exception:
            continue  # Silently skip structures that throw parsing errors

    df_lip = pd.DataFrame({
        'MW': mw,
        'LogP': logp,
        'NumHDonors': h_donors,
        'NumHAcceptors': h_acceptors
    })
    return df_lip, valid_indices

# Load raw data and drop structurally missing targets
df_raw = pd.read_csv('bioactivity_data_raw.csv').dropna(subset=['canonical_smiles', 'standard_value'])

# Compute properties only on structurally valid compounds
lipinski_df, valid_rows = calculate_lipinski_safe(df_raw['canonical_smiles'])
df_filtered = df_raw.iloc[valid_rows].reset_index(drop=True)

# Combine datasets cleanly without leaving internal NaN rows
df_processed = pd.concat([df_filtered, lipinski_df], axis=1)

# Safely handle IC50 conversions to prevent log10(0) math errors
bio_val = pd.to_numeric(df_processed['standard_value'], errors='coerce')
df_processed['pIC50'] = -np.log10(bio_val.clip(lower=1e-10) * (10**-9))

# Drop any remaining unparseable anomalies
df_processed = df_processed.dropna(subset=['pIC50', 'MW'])
df_processed.to_csv('bioactivity_data_processed.csv', index=False)

print(f"Data Processing Complete. Remaining usable compounds for ML: {len(df_processed)}")


Data Processing Complete. Remaining usable compounds for ML: 262


In [13]:
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Compute fingerprints
def get_fingerprints(smiles_list):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
        fps.append(np.array(fp))
    return np.array(fps)

X = get_fingerprints(df_processed['canonical_smiles'])
y = df_processed['pIC50'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Predictor
model = LGBMRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
preds = model.predict(X_test)
print(f"Model Performance -> R2 Score: {r2_score(y_test, preds):.2f} | MSE: {mean_squared_error(y_test, preds):.2f}")


[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerator
[05:17:47] DEPRECATION WARNING: please use MorganGenerat

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000679 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 260
[LightGBM] [Info] Number of data points in the train set: 209, number of used features: 130
[LightGBM] [Info] Start training from score 5.023191
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerator
[05:17:49] DEPRECATION WARNING: please use MorganGenerat

In [15]:
import urllib.request

# Download target PDB (Example: SARS-CoV-2 Mpro structure 6LU7)
pdb_id = "6LU7"
urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", "protein.pdb")

# Strip structural waters and heteroatoms using basic shell scripting
with open("protein.pdb", "r") as f:
    lines = f.readlines()

clean_protein = [l for l in lines if l.startswith("ATOM")]
with open("protein_clean.pdb", "w") as f:
    f.writelines(clean_protein)

# Convert clean PDB to PDBQT format for docking engine compatibility
!obabel protein_clean.pdb -O protein.pdbqt -xr

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is protein_clean.pdb)

1 molecule converted


In [18]:
import sys
!{sys.executable} -m pip install meeko # Temporary fix for ModuleNotFoundError
!{sys.executable} -m pip install gemmi # Install missing dependency for meeko

from meeko import MoleculePreparation
from rdkit import Chem
from rdkit.Chem import AllChem

# Top hit smile example
sample_smiles = "CC(=O)N[C@@H](CC1=CC=CC=C1)C(=O)N"
mol = Chem.MolFromSmiles(sample_smiles)
mol_h = Chem.AddHs(mol)

# Embed 3D Coordinates
AllChem.EmbedMolecule(mol_h, AllChem.ETKDG())
AllChem.MMFFOptimizeMolecule(mol_h)

# Convert to PDBQT with Meeko to map rotatable bonds
preparer = MoleculePreparation()
preparer.prepare(mol_h)
preparer.write_pdbqt_file("ligand.pdbqt")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 34.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/meeko/preparation.py:714: DeprecationWarning: MoleculePreparation.write_pdbqt_file() is deprecated since Meeko v0.5
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meeko/preparation.py:693: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/usr/local/lib/python3.12/dist-packages/meeko/preparation.py:467: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)


In [23]:
import numpy as np
from vina import Vina

def get_protein_center_and_size(pdbqt_file):
    """Automatically calculates the geometric center and box size for a protein."""
    coords = []
    with open(pdbqt_file, 'r') as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                # Extract X, Y, Z coordinates from fixed PDBQT format positions
                try:
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    coords.append([x, y, z])
                except ValueError:
                    continue

    coords = np.array(coords)
    if len(coords) == 0:
        raise ValueError("The protein.pdbqt file contains no structural coordinate data!")

    # Find spatial boundaries
    min_coords = coords.min(axis=0)
    max_coords = coords.max(axis=0)

    # Calculate geometric center and add a safety padding buffer to the box size
    center = ((min_coords + max_coords) / 2).tolist()
    size = (max_coords - min_coords + 12.0).tolist() # 12 Ångström total safety margin

    return center, size

# Extract dynamic spatial boundaries
center_coords, box_dims = get_protein_center_and_size('protein.pdbqt')
print(f"Calculated Target Center: {center_coords}")
print(f"Calculated Search Box Size: {box_dims}")

# Initialize docking simulation configuration
v = Vina(sf_name='vina')
v.set_receptor('protein.pdbqt')
v.set_ligand_from_file('ligand.pdbqt')

# Mount calculated structural grid maps
v.compute_vina_maps(center=center_coords, box_size=box_dims)

# Execute active docking simulation
print("Running simulation...")
v.dock(exhaustiveness=8, n_poses=5)

# Save configurations and parse output
v.write_poses('docking_results.pdbqt', overwrite=True)
scores = v.energies()

# Isolate top active binding value (index 0, first score item)
top_score = scores[0][0].item() # Ensure scalar conversion
print(f"✅ Top Binding Free Energy Hit: {top_score:.2f} kcal/mol")

Calculated Target Center: [-25.5435, 12.8535, 58.25]
Calculated Search Box Size: [62.669, 77.439, 71.20400000000001]
Running simulation...
✅ Top Binding Free Energy Hit: -6.32 kcal/mol


In [24]:
import pandas as pd

# Load structural values from your machine learning step (Step 4)
try:
    df_results = pd.read_csv('bioactivity_data_processed.csv')

    # Grab the top hits from your screening pool
    top_hit_smiles = df_results.sort_values(by='pIC50', ascending=False)['canonical_smiles'].iloc[0]
    top_hit_id = df_results.sort_values(by='pIC50', ascending=False)['molecule_chembl_id'].iloc[0]

    print("--- PIPELINE HIT DISCOVERY RESULTS ---")
    print(f"Top Candidate ID: {top_hit_id}")
    print(f"Top Candidate SMILES: {top_hit_smiles}")
    print(f"Calculated Affinity Score: {top_score:.2f} kcal/mol")

except FileNotFoundError:
    print("Warning: 'bioactivity_data_processed.csv' missing. Run Step 3 & 4 fixes first!")


--- PIPELINE HIT DISCOVERY RESULTS ---
Top Candidate ID: CHEMBL5565685
Top Candidate SMILES: CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CCNC3=O)N(C(=O)C3(c4cccc(Cl)c4)CC3)C[C@@H]21
Calculated Affinity Score: -6.32 kcal/mol


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [25]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Retrieve candidate info
smiles = "CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CCNC3=O)N(C(=O)C3(c4cccc(Cl)c4)CC3)C[C@@H]21"
mol = Chem.MolFromSmiles(smiles)

# Calculate key structural metrics
mw = Descriptors.MolWt(mol)
logp = Descriptors.MolLogP(mol)
h_donors = Chem.Lipinski.NumHDonors(mol)
h_acceptors = Chem.Lipinski.NumHAcceptors(mol)

# Structure the discovery passport
discovery_passport = {
    "Metric": [
        "Candidate ID",
        "SMILES String",
        "Binding Affinity",
        "Molecular Weight",
        "LogP (Hydrophobicity)",
        "H-Bond Donors",
        "H-Bond Acceptors"
    ],
    "Value": [
        "CHEMBL5565685",
        smiles,
        "-6.32 kcal/mol",
        f"{mw:.2f} g/mol",
        f"{logp:.2f}",
        str(h_donors),
        str(h_acceptors)
    ]
}

df_passport = pd.DataFrame(discovery_passport)
df_passport.to_csv("lead_candidate_passport.csv", index=False)

# Display the dashboard
print("==================================================")
print("           FINAL LEAD CANDIDATE REPORT            ")
print("==================================================")
print(df_passport.to_string(index=False))
print("==================================================")
print("🎉 Success! Pipeline complete. 'lead_candidate_passport.csv' saved.")


           FINAL LEAD CANDIDATE REPORT            
               Metric                                                                                    Value
         Candidate ID                                                                            CHEMBL5565685
        SMILES String CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CCNC3=O)N(C(=O)C3(c4cccc(Cl)c4)CC3)C[C@@H]21
     Binding Affinity                                                                           -6.32 kcal/mol
     Molecular Weight                                                                             471.99 g/mol
LogP (Hydrophobicity)                                                                                     2.06
        H-Bond Donors                                                                                        2
     H-Bond Acceptors                                                                                        4
🎉 Success! Pipeline complete. 'lead_candidate_passport.csv' s

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [29]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation
from vina import Vina

# The original top hit SMILES (contains Cl at the desired substitution site)
# This variable is available from the kernel state as 'top_hit_smiles'
top_hit_smiles = "CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CCNC3=O)N(C(=O)C3(c4cccc(Cl)c4)CC3)C[C@@H]21"
core_mol_cl = Chem.MolFromSmiles(top_hit_smiles)

# Define functional groups to substitute the Chlorine with
# These are simple molecules, not fragments with attachment points
substituents = {
    "Fluorine (-F)": Chem.MolFromSmiles("F"),
    "Methyl (-CH3)": Chem.MolFromSmiles("C"),
    "Trifluoromethyl (-CF3)": Chem.MolFromSmiles("C(F)(F)F"),
    "Hydroxyl (-OH)": Chem.MolFromSmiles("O"),
    "Nitrile (-CN)": Chem.MolFromSmiles("C#N")
}

optimization_results = []

# Fetch the spatial grid constraints calculated in Step 7
# (Ensure center_coords and box_dims from your previous step are still in memory)

for name, sub_mol in substituents.items():
    print(f"Mutating and testing variant: {name}...")

    # Define the reaction SMARTS to replace the Chlorine atom in the core with the substituent.
    # [c:1]Cl - matches a carbon atom (with map 1) bonded to a Chlorine. This matches the core_mol_cl.
    # [*:2] - matches any atom/group in the substituent molecule. The first atom of the substituent will be mapped to *:2.
    # >>[c:1][*:2] - The Chlorine is replaced by the substituent's first atom/group.
    # This reaction ensures the Cl is removed and the new group is attached.
    rxn = AllChem.ReactionFromSmarts('[c:1]Cl.[*:2]>>[c:1][*:2]')

    products = rxn.RunReactants((core_mol_cl, sub_mol))

    if not products:
        print(f"  Warning: No product generated for {name}. Skipping.")
        continue

    # Take the first product (assuming only one is generated and it's valid)
    product = products[0][0]
    Chem.SanitizeMol(product) # Sanitize the new molecule
    variant_smiles = Chem.MolToSmiles(product)

    # 3D Geometry Generation
    mol_h = Chem.AddHs(product)
    AllChem.EmbedMolecule(mol_h, AllChem.ETKDG()) # Use ETKDG for better conformers
    AllChem.MMFFOptimizeMolecule(mol_h)

    # Prepare PDBQT format file
    preparer = MoleculePreparation()
    preparer.prepare(mol_h)
    preparer.write_pdbqt_file("variant.pdbqt")

    # Execute Docking Vina
    v = Vina(sf_name='vina')
    v.set_receptor('protein.pdbqt')
    v.set_ligand_from_file('variant.pdbqt')
    v.compute_vina_maps(center=center_coords, box_size=box_dims)
    v.dock(exhaustiveness=4, n_poses=1) # Accelerated settings for screening

    # Extract calculated energy score
    # v.energies() returns a list of tuples, each (binding_energy, rmsd_to_centroid, rmsd_to_reference)
    # We want the binding_energy from the first pose
    score = v.energies()[0][0] # Access the first element (binding_energy) of the first pose
    optimization_results.append({
        "Variant": name,
        "SMILES": variant_smiles,
        "Binding Affinity (kcal/mol)": round(score, 2)
    })

# Compile optimization tracking table
df_opt = pd.DataFrame(optimization_results)
df_opt = df_opt.sort_values(by="Binding Affinity (kcal/mol)").reset_index(drop=True)

print("\n==================================================")
print("         STRUCTURAL OPTIMIZATION REPORT          ")
print("==================================================")
print(df_opt.to_string(index=False))
print("==================================================")

Mutating and testing variant: Fluorine (-F)...


/usr/local/lib/python3.12/dist-packages/meeko/preparation.py:714: DeprecationWarning: MoleculePreparation.write_pdbqt_file() is deprecated since Meeko v0.5
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meeko/preparation.py:693: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/usr/local/lib/python3.12/dist-packages/meeko/preparation.py:467: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)


Mutating and testing variant: Methyl (-CH3)...
Mutating and testing variant: Trifluoromethyl (-CF3)...
Mutating and testing variant: Hydroxyl (-OH)...
Mutating and testing variant: Nitrile (-CN)...

         STRUCTURAL OPTIMIZATION REPORT          
               Variant                                                                                         SMILES  Binding Affinity (kcal/mol)
Trifluoromethyl (-CF3) CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CCNC3=O)N(C(=O)C3(c4cccc(C(F)(F)F)c4)CC3)C[C@@H]21                        -7.32
         Methyl (-CH3)            Cc1cccc(C2(C(=O)N3C[C@H]4[C@@H]([C@H]3C(=O)N[C@H](C=O)C[C@@H]3CCNC3=O)C4(C)C)CC2)c1                        -7.25
         Fluorine (-F)        CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CCNC3=O)N(C(=O)C3(c4cccc(F)c4)CC3)C[C@@H]21                        -7.09
        Hydroxyl (-OH)        CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CCNC3=O)N(C(=O)C3(c4cccc(O)c4)CC3)C[C@@H]21                        -6.52
         Nitrile

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [30]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Fragments, Crippen

# Optimized Lead SMILES (-CF3 Variant)
opt_smiles = "CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CCNC3=O)N(C(=O)C3(c4cccc(C(F)(F)F)c4)CC3)C[C@@H]21"
mol = Chem.MolFromSmiles(opt_smiles)

# 1. Absorption Indicators
logp = Crippen.MolLogP(mol) # Octanol-water partition coefficient
tpsa = Descriptors.TPSA(mol) # Topological Polar Surface Area (Ideally < 140 Å² for gut absorption)

# 2. Structural Alerts (Pan-Assay Interference Compounds - PAINS)
# We check for reactive/promiscuous functional groups like aldehydes
# Your molecule contains an aldehyde, which is a known reactive warhead.
has_aldehyde = Fragments.fr_aldehyde(mol) > 0

# 3. Synthetic Accessibility Estimate (Basic structural complexity check)
num_chiral_centers = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
num_rings = Descriptors.RingCount(mol)

# Determine basic safety status
adme_status = "Pass"
warnings = []
if tpsa > 140:
    warnings.append("High TPSA (Poor membrane permeability)")
if logp > 5:
    warnings.append("High LogP (Poor water solubility)")
if has_aldehyde:
    warnings.append("Covalent Warhead Detected (Aldehyde present - common for protease inhibitors)")

if len(warnings) > 0:
    adme_status = f"Conditional Pass ({', '.join(warnings)})"

# Build ADMET Report DataFrame
admet_data = {
    "ADMET Parameter": [
        "Optimized SMILES",
        "LogP (Lipophilicity)",
        "TPSA (Polar Surface Area)",
        "Chiral Center Count",
        "Ring Count",
        "Reactive Group Alert",
        "Overall ADMET Profile Status"
    ],
    "Value": [
        opt_smiles,
        f"{logp:.2f} (Target: < 5.0)",
        f"{tpsa:.2f} Å² (Target: < 140 Å²)",
        f"{num_chiral_centers} (High stereochemical complexity)",
        f"{num_rings}",
        "Yes (Aldehyde warhead)" if has_aldehyde else "None",
        adme_status
    ]
}

df_admet = pd.DataFrame(admet_data)
df_admet.to_csv("optimized_lead_admet.csv", index=False)

print("\n==================================================")
print("             ADMET PROFILE REPORT                ")
print("==================================================")
print(df_admet.to_string(index=False))
print("==================================================")



             ADMET PROFILE REPORT                
             ADMET Parameter                                                                                            Value
            Optimized SMILES   CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CCNC3=O)N(C(=O)C3(c4cccc(C(F)(F)F)c4)CC3)C[C@@H]21
        LogP (Lipophilicity)                                                                             2.43 (Target: < 5.0)
   TPSA (Polar Surface Area)                                                                      95.58 Å² (Target: < 140 Å²)
         Chiral Center Count                                                               5 (High stereochemical complexity)
                  Ring Count                                                                                                5
        Reactive Group Alert                                                                           Yes (Aldehyde warhead)
Overall ADMET Profile Status Conditional Pass (Covalent Warhead Det

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:
!pip install -q py3Dmol
